# BiRefNet Background Removal

This notebook loads the official BiRefNet model once and removes the background from an input image.

In [ ]:
!pip install -q transformers torch torchvision pillow opencv-python

In [ ]:
from PIL import Image
import torch
from transformers import AutoModelForImageSegmentation
from torchvision import transforms

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "ZhengPeng7/BiRefNet"

model = AutoModelForImageSegmentation.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model.to(DEVICE)
model.eval()

print("Loaded on:", DEVICE)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    ),
])

In [ ]:
def remove_background(image_path, output_path="transparent.png"):
    image = Image.open(image_path).convert("RGB")
    original_size = image.size

    input_tensor = transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(input_tensor)[-1].sigmoid().cpu()[0][0]

    pred = transforms.ToPILImage()(pred)
    pred = pred.resize(original_size)

    image = image.convert("RGBA")
    image.putalpha(pred)

    image.save(output_path)

    return image

In [ ]:
def save_white_background(rgba_image, output_path="white_bg.png"):
    white = Image.new("RGBA", rgba_image.size, (255,255,255,255))
    white.paste(rgba_image, mask=rgba_image.split()[-1])
    white.convert("RGB").save(output_path)

In [ ]:
IMAGE_PATH = "input.jpg"   # Change to your image path

rgba = remove_background(
    IMAGE_PATH,
    "transparent.png"
)

save_white_background(
    rgba,
    "white_bg.png"
)

print("Done!")
print("Generated:")
print("- transparent.png")
print("- white_bg.png")